In [1]:
# імпорт базових модулей
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
from tqdm import tqdm
from pathlib import Path
import shutil
from pprint import pprint

# параметри виведення
pd.set_option("display.max_columns", 500) # кількість колонок
pd.set_option("display.max_rows", 1000) # кількість рядків
pd.set_option("display.max_colwidth", 300) # ширина колонок
pd.set_option("display.precision", 5) # кількість знаків після коми

# вимикаємо зайві попередження
import warnings
warnings.filterwarnings("ignore")

# друк всіх результатів в одній комірці а не тільки останнього
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# магічний метод для того щоб отримувати графіки біля комірок з кодом
# %matplotlib inline

In [5]:
# імпорт додаткових модулей

# import sqlite3
# import pyarrow as pa
# # from pyarrow import csv
# import pyarrow.parquet as pq
# import math

from sentence_transformers import SentenceTransformer

In [2]:
# data_dir = "data"
data_file = "data/arxiv_subset.parquet"
df = pd.read_parquet(data_file)
df['title+abstract'] = df['title'] + " [SEP] " + df['abstract']
df.info()
df.head(5)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id              10000 non-null  str  
 1   title           10000 non-null  str  
 2   abstract        10000 non-null  str  
 3   authors         10000 non-null  str  
 4   year            10000 non-null  int64
 5   category        10000 non-null  str  
 6   title+abstract  10000 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.7 MB


,id,title,abstract,authors,year,category,title+abstract
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and LHC energies,"A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions from quark-antiquark, gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as...","BalázsC., BergerE. L., NadolskyP. M., YuanC. -P.",2007,hep-ph,Calculation of prompt diphoton production cross sections at Tevatron and LHC energies [SEP] A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions ...
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use it obtain a characterization of the family of $(k,\ell)$-sparse graphs and algorithmic solutions to a family of problems concerning tree decompositions of graphs. Special instances of sparse graphs appear in rigidity th...","StreinuIleana, TheranLouis",2007,math.CO,"Sparsity-certifying Graph Decompositions [SEP] We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use it obtain a characterization of the family of $(k,\ell)$-sparse graphs and algorithmic solutions to a family of problems concerning tree decompositions of graphs. Special i..."
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field fluid model,"The evolution of Earth-Moon system is described by the dark matter field fluid model proposed in the Meeting of Division of Particle and Field 2004, American Physical Society. The current behavior of the Earth-Moon system agrees with this model very well and the general pattern of the evolution ...",PanHongjun,2007,physics.gen-ph,"The evolution of the Earth-Moon system based on the dark matter field fluid model [SEP] The evolution of Earth-Moon system is described by the dark matter field fluid model proposed in the Meeting of Division of Particle and Field 2004, American Physical Society. The current behavior of the Ea..."
3,0704.0004,A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata,We show that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata. The proof involves a bijection from these automata to certain marked lattice paths and a sign-reversing involution to evaluate the determinant.,CallanDavid,2007,math.CO,A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata [SEP] We show that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata. The proof involves a bijection from these automata to certain marked lattice paths and a sign-reve...
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$,"In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge 0$, using the dyadic grid. This result is a consequence of the description of the Hardy spaces $H^p(R^N)$ in terms of dyadic and special atoms.","Abu-ShammalaWael, TorchinskyAlberto",2007,math.CA,"From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$ [SEP] In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge 0$, using the dyadic grid. This result is a consequence of the description of the Hardy spaces $H^p(R^N)$ in terms of dyadic and special atoms."


In [3]:
INPUT_FILE = "data/arxiv_subset.parquet"
OUTPUT_FILE = "embeddings/arxiv_embeddings.parquet"
# MODEL_NAME = "allenai/specter2_base"
MODEL_NAME = "sentence-transformers/allenai-specter"
BATCH_SIZE = 64

In [10]:
model = SentenceTransformer(MODEL_NAME)

In [17]:
def make_text(row: pd.Series) -> str:
    title = str(row["title"]).strip()
    abstract = str(row["abstract"]).strip()
    return f"{title} [SEP] {abstract}"


texts = [make_text(row) for _, row in df.iterrows()]

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

df["embedding"] = embeddings.tolist()
# df.to_parquet(OUTPUT_FILE, index=False)
# print(f"Saved {len(df)} embeddings to {OUTPUT_FILE}")

Batches:   1%|          | 2/313 [02:39<6:52:13, 79.53s/it]


KeyboardInterrupt: 

In [15]:
embeddings

NameError: name 'embeddings' is not defined

In [14]:
    df["embedding"]
    

KeyError: 'embedding'

In [2]:
matrix = np.load('embeddings/embeddings.npy')
matrix.shape

(10000, 768)

In [3]:
matrix[0]

array([ 1.98680088e-02,  3.86331901e-02,  1.44202067e-02,  2.46090312e-02,
       -8.91747046e-03, -1.77954212e-02,  6.16089366e-02,  2.95052025e-02,
        1.57969613e-02, -6.06374303e-03,  2.13780161e-02, -2.19329447e-02,
        3.79580781e-02,  1.11837557e-03,  9.26020276e-03, -2.23652981e-02,
       -2.32699164e-03, -1.34612042e-02,  3.13074552e-02, -5.83511591e-03,
        2.56701768e-03, -9.88954026e-03, -4.02686559e-02,  2.48887185e-02,
       -3.40341888e-02,  5.74985147e-02, -2.28009764e-02,  3.60187367e-02,
       -2.18245853e-03,  1.38508501e-02,  2.11142357e-02, -3.55164744e-02,
        3.65358442e-02, -4.38768677e-02,  1.44484011e-03, -4.63505238e-02,
       -1.13630341e-02, -1.34549066e-02, -2.34865248e-02,  4.75003421e-02,
        3.14082317e-02, -1.37108630e-02,  1.04653155e-02, -3.36675607e-02,
       -1.40884193e-02,  3.20932567e-02,  1.59231685e-02,  3.68388966e-02,
        2.93653291e-02, -3.10645849e-02,  3.24346907e-02, -5.99899329e-02,
        5.23118675e-03,  

In [4]:
documents = pd.read_parquet("data/arxiv_subset.parquet")
documents.shape
documents.info()
documents.head(5)

(10000, 6)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        10000 non-null  str  
 1   title     10000 non-null  str  
 2   abstract  10000 non-null  str  
 3   authors   10000 non-null  str  
 4   year      10000 non-null  int64
 5   category  10000 non-null  str  
dtypes: int64(1), str(5)
memory usage: 9.3 MB


,id,title,abstract,authors,year,category
0,0704.0001,Calculation of prompt diphoton production cross sections at Tevatron and LHC energies,"A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions from quark-antiquark, gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as...","BalázsC., BergerE. L., NadolskyP. M., YuanC. -P.",2007,hep-ph
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use it obtain a characterization of the family of $(k,\ell)$-sparse graphs and algorithmic solutions to a family of problems concerning tree decompositions of graphs. Special instances of sparse graphs appear in rigidity th...","StreinuIleana, TheranLouis",2007,math.CO
2,0704.0003,The evolution of the Earth-Moon system based on the dark matter field fluid model,"The evolution of Earth-Moon system is described by the dark matter field fluid model proposed in the Meeting of Division of Particle and Field 2004, American Physical Society. The current behavior of the Earth-Moon system agrees with this model very well and the general pattern of the evolution ...",PanHongjun,2007,physics.gen-ph
3,0704.0004,A determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata,We show that a determinant of Stirling cycle numbers counts unlabeled acyclic single-source automata. The proof involves a bijection from these automata to certain marked lattice paths and a sign-reversing involution to evaluate the determinant.,CallanDavid,2007,math.CO
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$,"In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge 0$, using the dyadic grid. This result is a consequence of the description of the Hardy spaces $H^p(R^N)$ in terms of dyadic and special atoms.","Abu-ShammalaWael, TorchinskyAlberto",2007,math.CA


In [5]:
len(documents)

10000

In [7]:
vectors_to_upsert = [
    {
        "id": documents.iloc[doc_idx]["id"],
        "values": matrix[doc_idx].tolist(),
        "metadata": {key: value for key, value in documents.iloc[doc_idx].items() if pd.notnull(value)},
    }
    for doc_idx in tqdm(range(0, 20), desc="Підготовка векторів до upsert")
]

Підготовка векторів до upsert: 100%|██████████| 20/20 [00:00<00:00, 3856.83it/s]


In [8]:
vectors_to_upsert[0]

{'id': '0704.0001',
 'values': [0.01986800879240036,
  0.03863319009542465,
  0.014420206658542156,
  0.024609031155705452,
  -0.008917470462620258,
  -0.017795421183109283,
  0.06160893663764,
  0.029505202546715736,
  0.01579696126282215,
  -0.006063743028789759,
  0.021378016099333763,
  -0.021932944655418396,
  0.03795807808637619,
  0.001118375570513308,
  0.009260202758014202,
  -0.022365298122167587,
  -0.002326991641893983,
  -0.013461204245686531,
  0.03130745515227318,
  -0.005835115909576416,
  0.0025670176837593317,
  -0.009889540262520313,
  -0.040268655866384506,
  0.024888718500733376,
  -0.03403418883681297,
  0.0574985146522522,
  -0.022800976410508156,
  0.0360187366604805,
  -0.0021824585273861885,
  0.01385085005313158,
  0.021114235743880272,
  -0.03551647439599037,
  0.03653584420681,
  -0.04387686774134636,
  0.0014448401052504778,
  -0.046350523829460144,
  -0.011363034136593342,
  -0.013454906642436981,
  -0.02348652482032776,
  0.04750034213066101,
  0.0314082

In [2]:
df = pd.read_parquet("data/arxiv_subset.parquet")
df['abstract_length'] = df['abstract'].fillna("").str.len()

df.sort_values(by='abstract_length', ascending=False, inplace=True)

In [3]:
df.iloc[0:30]

,id,title,abstract,authors,year,category,abstract_length
7848,0705.3846,The SN 1987A Link to Gamma-Ray Bursts,"Early measurements of SN 1987A indicate a beam/jet (BJ) which hit polar ejecta (PE) to produce the ""Mystery Spot"" (MS). The SN flash takes an extra 8 d to hit the MS, and this was confirmed at 2e39 ergs/s in the optical at day 8. A ramp in luminosity starting near day 10 indicates particles from...",MiddleditchJohn,2007,astro-ph,1872
987,0704.0988,Evidence for a Massive Protocluster in S255N,S255N is a luminous far-infrared source that contains many indications of active star formation but lacks a prominent near-infrared stellar cluster. We present mid-infrared through radio observations aimed at exploring the evolutionary state of this region. Our observations include 1.3mm continu...,"CyganowskiC. J., BroganC. L., HunterT. R.",2007,astro-ph,1866
1300,0704.1301,IRAS 18317-0757: A Cluster of Embedded Massive Stars and Protostars,"We present high-resolution, multiwavelength continuum and molecular-line images of the massive star forming region IRAS 18317-0757. The IR through mm spectral energy distribution can be approximated by a two-temperature model (25 and 63 K) with a total luminosity of approximately log(L/Lsun)=5.2...","HunterT. R., ZhangQ., SridharanT. K.",2007,astro-ph,1858
2444,0704.2445,Multi-Color Photometry of the Galactic Globular Cluster M75 = NGC 6864. A New Sensitive Metallicity Indicator and the Position of the Horizontal Branch in UV,"We carry out and analyze new multi-color photometry of the Galactic globular cluster (GC) M75 in UBVI and focus on the brighter sequences of the color- magnitude diagram (CMD), with particular emphasis on their location in U-based CMD. Specifically, we study the level both of the horizontal (HB)...","KravtsovV., AlcainoG., MarconiG., AlvaradoF.",2007,astro-ph,1857
8345,0705.4343,A Systematic Study of the Final Masses of Gas Giant Planets,"We construct an analytic model for the rate of gas accretion onto a planet embedded in a protoplanetary disk as a function of planetary mass, disk viscosity, disk scale height, and unperturbed surface density in order to study the long-term accretion and final masses of gas giant planets. We fir...","TanigawaT., IkomaM.",2007,astro-ph,1854
2195,0704.2196,Absolute Calibration and Characterization of the Multiband Imaging Photometer for Spitzer. II. 70 micron Imaging,The absolute calibration and characterization of the Multiband Imaging Photometer for Spitzer (MIPS) 70 micron coarse- and fine-scale imaging modes are presented based on over 2.5 years of observations. Accurate photometry (especially for faint sources) requires two simple processing steps beyon...,"GordonKarl D., EngelbrachtCharles W., FaddaDario, StansberryJohn, WachterStefanie, FrayerDave T., RiekeGeorge, Noriega-CrespoAlberto, LatterWilliam B., YoungErick",2007,astro-ph,1853
798,0704.0799,Spin Evolution of Accreting Neutron Stars: Nonlinear Development of the R-mode Instability,The nonlinear saturation of the r-mode instability and its effects on the spin evolution of Low Mass X-ray Binaries (LMXBs) are modeled using the triplet of modes at the lowest parametric instability threshold. We solve numerically the coupled equations for the three mode amplitudes in conjuncti...,"BondarescuRuxandra, TeukolskySaul A., WassermanIra",2007,astro-ph,1851
3099,0704.3100,The Origin of the Galaxy Mass-Metallicity Relation and Implications for Galactic Outflows,"(Abridged) Using cosmological hydrodynamic simulations in combination with analytic modeling, we show that the galaxy stellar mass-metallicity relation (MZR) provides strong constraints on galactic outflows across cosmic time. We compare three outflow models: No outflows, a ""constant wind"" (cw) ...","FinlatorK., DaveR.",2007,astro-ph,1849
5208,0705.1206,The SSS phase of RS Ophiuchi observed with Chandra and XMM-Newton I.: Data and preliminary Modeling,"The phase of Super-Soft-Source (SSS) emission of the sixth recorded outbu